# 머신러닝 파일별 1 · 02 Iris 첫 모델과 모델 교체

**한 줄 요약** · Iris 150송이를 학습 120 · 시험 30으로 나누고, `fit → predict → score` 같은 틀로 모델 3개를 바꿔 끼우며 `random_state`가 결과를 고정하는 이유까지 확인

- **원본** · `6.머닝러신/01_머신러닝_입문_Iris/02_iris_quickstart.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-1 1장 (분할 · 정확도) · 3장 (seed)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `load_iris()`가 돌려주는 X(150, 4) · y(150,)의 뜻을 말한다
2. 학습용과 시험용을 나누는 이유를 '같은 문제로 공부하고 시험 보기'로 설명한다
3. `fit → predict → accuracy_score` 세 단계로 첫 모델을 학습 · 평가한다
4. 모델 이름만 바꿔 DecisionTree · LogisticRegression · KNN을 같은 틀로 비교한다
5. `random_state`를 고정해야 결과를 다시 만들 수 있는 이유를 실행으로 보인다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **새 데이터로 첫 모델을 돌려 볼 때** · `fit → predict → score` 틀 (2장)
- **'정확도 100%'라는 결과를 받았을 때** · 시험 데이터가 30개뿐인지 · seed가 고정됐는지 (3장)
- **여러 모델 중 무엇을 쓸지 정할 때** · 같은 분할에서 모델만 바꿔 비교 (2-3)
- **다른 사람이 내 결과를 다시 만들어야 할 때** · `random_state` 고정 (3-2)

## 학습 지도

<img src="images/pfm1_map.png" alt="02_iris_quickstart 학습 지도" width="763">

**예상 학습 시간** · 개념 30분 + 실습 30분 (전체 실행 10초 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 원본 목표</td></tr>
<tr><td>1</td><td>데이터 준비</td><td>Iris 데이터는 어떻게 생겼고, 왜 학습용과 시험용으로 나누는가?</td></tr>
<tr><td>2</td><td>첫 모델과 모델 교체</td><td>모델은 어떤 세 단계로 쓰고, 모델을 바꾸면 무엇만 달라지는가?</td></tr>
<tr><td>3</td><td>random_state</td><td><code>random_state</code>를 바꾸거나 빼면 왜 결과가 달라지고, 어떻게 고정하는가?</td></tr>
<tr><td>4</td><td>종합 실습</td><td>분할 운을 줄이면서 모델 3개를 공정하게 비교하려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

이 노트북은 scikit-learn에 들어 있는 Iris 데이터만 씀 (파일 필요 없음). **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · scikit-learn · numpy · pandas (원본 버전 sklearn 1.6.1)
- **seed** · `random_state=42` (원본과 같음) · 첫 트리만 원본대로 18
- **데이터** · `load_iris()` · 150송이 × 측정값 4개 · 품종 3개
- **주의** · sklearn 버전이 다르면 정확도 끝자리가 원본과 다를 수 있음

**실행 흐름**

1. `sklearn` · `numpy` · `pandas` · `matplotlib` 불러오기
2. 버전 출력 (다르면 정확도가 조금 다를 수 있음)

In [1]:
# 실행 환경 확인: 버전이 다르면 정확도 숫자가 조금 다를 수 있음
# 왜 · 버전에 따라 정확도가 조금 달라질 수 있어 실행 환경을 기록해 두기 위해
import matplotlib    # 그래프 (그림은 images/에 미리 저장)
import numpy as np   # 배열 계산
import pandas as pd  # 표로 보기
import seaborn       # 원본 산점도용 (버전만 확인)
import sklearn       # 머신러닝 라이브러리

print(f"sklearn: {sklearn.__version__}")                                         # 원본은 1.6.1
print(f"numpy: {np.__version__} · pandas: {pd.__version__}")                     # 배열 · 표
print(f"matplotlib: {matplotlib.__version__} · seaborn: {seaborn.__version__}")  # 그래프

sklearn: 1.9.0
numpy: 2.4.6 · pandas: 3.0.3
matplotlib: 3.11.0 · seaborn: 0.13.2


### 원본 목표

- 데이터를 학습 · 시험용으로 나누는 이유를 이해
- 첫 모델(DecisionTree)로 학습 → 예측 → 정확도를 계산
- 같은 절차로 LogisticRegression, KNN으로 교체
- 간단한 파라미터 변경으로 성능 차이를 직접 확인

**원본 힌트**

- 모델 코드는 3줄이면 충분 · `fit(학습) → predict(예측) → score(평가)`
- 돌리기는 쉽지만 성능을 올리기는 어려움 → EDA · 전처리 · 하이퍼파라미터 튜닝 · 모델 이해가 중요

---
## 1. 데이터 준비

> **이 장의 흐름**
>
> - 데이터를 불러와 X · y 모양 확인 → 표로 펼쳐 보기
> - 그다음 학습 120 · 시험 30으로 나눔 (품종 비율 유지)

<img src="images/pfm1_scatter.png" alt="꽃받침 길이 · 너비 산점도 (원본 셀 9)" width="588">

**파이프라인 위치** · ① 데이터 준비 → ② 분할

### 1-1. `A` `load_iris()`로 X · y 받기 · `return_X_y=True` · `.shape` <sub>원본 셀 4 ~ 7</sub>

> 🎯 **`load_iris()`가 주는 X와 y는 각각 무엇이고 모양은 어떻게 되는가?**

💡 **쉽게 말하면**

- X는 **문제지**(측정값 4개), y는 **정답지**(품종 번호).
- 꽃 150송이 → X는 150행 4열, y는 150개.

**실행 흐름**

1. `return_X_y=True`로 X · y 받기
2. 모양 확인
3. 전체 묶음(Bunch)의 열쇠 이름 보기

**먼저 예측** · X와 y의 shape는? — 답은 코드 실행 결과 아래에

In [2]:
from sklearn.datasets import load_iris  # 내장 Iris 데이터

# 왜 · 모델에 바로 넣을 특성 X와 정답 y를 한 번에 받기 위해
X, y = load_iris(return_X_y=True)  # 문제(X)와 정답(y)만 바로 받기
print("X shape:", X.shape)         # (샘플 수, 특성 수)
print("y shape:", y.shape)         # 샘플마다 정답 1개

# 왜 · 숫자뿐 아니라 열 이름 · 클래스 이름 같은 설명 정보도 함께 보기 위해
iris = load_iris()                                       # 전체 묶음(Bunch) · 원본 셀 6 · 7
print("열쇠:", list(iris.keys())[:6])                    # 원본은 묶음 전체를 출력 → 열쇠만
print("첫 꽃:", iris.data[0], "→ 품종", iris.target[0])  # 한 샘플 = 측정값 4개 + 정답

X shape: (150, 4)
y shape: (150,)
열쇠: ['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names']
첫 꽃: [5.1 3.5 1.4 0.2] → 품종 0


**결과** · 예측 정답은 X `(150, 4)` · y `(150,)`

- `X shape: (150, 4)` — 꽃 150송이 × 측정값 4개 (꽃받침 길이 · 너비, 꽃잎 길이 · 너비)
- `y shape: (150,)` — 송이마다 품종 번호 하나
- `열쇠: ['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names']` — `data` = X, `target` = y, 이름표는 `feature_names` · `target_names`
- `첫 꽃: [5.1 3.5 1.4 0.2] → 품종 0` — 첫 샘플은 setosa(0)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>load_iris(return_X_y=True)</code></td><td>X · y 배열 두 개만</td><td>바로 학습할 때 (기본 선택)</td></tr>
<tr><td><code>load_iris()</code></td><td>data · target · 이름표 · 설명까지 든 묶음</td><td>열 이름 · 품종 이름이 필요할 때</td></tr>
<tr><td><code>load_iris(as_frame=True)</code></td><td>pandas 표로 받음</td><td>표로 탐색(EDA)부터 할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 6 · 7은 같은 코드(`iris` 전체 출력)가 두 번 있음 → 한 번만 실행하면 됨
> - 묶음 전체를 출력하면 150행이 화면을 덮음 → 열쇠 · 첫 행만 확인

> ✅ **한 줄 정리** · `load_iris(return_X_y=True)`는 문제 X(150, 4)와 정답 y(150,)를 주고, 묶음에는 이름표까지 들어 있다.

**딥러닝 코드에서는**

- `images, labels = next(iter(loader))` — 딥러닝도 입력(X)과 정답(y)을 한 쌍으로 받음
- `images.shape` → `(배치, 채널, 높이, 너비)` · 첫 축이 샘플 수인 것은 같음

### 1-2. `B` 표로 펼쳐 보기 · `pd.DataFrame` · `target_names` · `value_counts()` <sub>원본 셀 8 · 9</sub>

> 🎯 **숫자 배열을 표로 바꾸면 무엇을 바로 확인할 수 있는가?**

💡 **쉽게 말하면**

- 배열은 숫자만 보이지만, 표로 바꾸면 **열 이름**과 **품종 개수**가 한눈에 보임.
- 품종 3개가 50송이씩 고르게 들어 있음.

**실행 흐름**

1. 품종 번호 · 이름 확인
2. DataFrame으로 바꾸고 label 열 추가
3. 앞 3행 · 품종별 개수 보기

**먼저 예측** · 품종 0 · 1 · 2는 각각 몇 송이일까? — 답은 코드 실행 결과 아래에

In [3]:
import pandas as pd  # 표 만들기

# 왜 · 숫자 라벨이 어떤 품종인지 표로 확인해 결과를 사람이 읽을 수 있게 하기 위해
print("iris target명:", iris.target_names)                          # 0 · 1 · 2의 이름
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)  # 측정값 4열 표
iris_df["label"] = iris.target                                      # 정답 열 추가
print(iris_df.head(3).to_string())                                  # 앞 3행
print(iris_df["label"].value_counts().sort_index().to_dict())       # 품종별 개수

iris target명: ['setosa' 'versicolor' 'virginica']
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  label
0                5.1               3.5                1.4               0.2      0
1                4.9               3.0                1.4               0.2      0
2                4.7               3.2                1.3               0.2      0
{0: 50, 1: 50, 2: 50}


**결과** · 예측 정답은 50 · 50 · 50 — 고르게 들어 있음

- `iris target명: ['setosa' 'versicolor' 'virginica']` — 번호 0 · 1 · 2의 품종 이름
- `sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  label` — 열 4개(cm 단위) + 정답 label
- `{0: 50, 1: 50, 2: 50}` — 품종마다 50송이 → 정확도가 착시를 덜 일으키는 균형 데이터

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>iris.data</code> (배열)</td><td>숫자만 · 모델에 바로 넣을 수 있음</td><td>모델에 넣을 때</td></tr>
<tr><td><code>pd.DataFrame(...)</code></td><td>열 이름 · 정렬 · 개수 세기</td><td>데이터를 눈으로 탐색할 때</td></tr>
<tr><td>산점도 (원본 셀 9 · 1장 그림)</td><td>두 특성의 분포를 품종별 색으로</td><td>특성이 품종을 가르는지 볼 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 8은 `iris` · `iris_data` · `iris_label`을 이름만 적어 두었는데, 셀 중간 줄은 출력되지 않음
> - 결과는 마지막 줄과 `print`만 화면에 나옴

> ✅ **한 줄 정리** · 배열을 DataFrame으로 바꾸면 열 이름과 품종별 개수(50 · 50 · 50)를 바로 확인할 수 있다.

**딥러닝 코드에서는**

- `Counter(dataset.targets)` — 이미지 데이터셋도 클래스 개수부터 셈
- 클래스가 불균형하면 정확도만 보지 않음

### 1-3. `A` `train_test_split`으로 나누기 · `test_size=0.2` · `stratify=y` <sub>원본 셀 10 · 11</sub>

> 🎯 **왜 나누고, `test_size` · `stratify` · `random_state`는 각각 무엇을 정하는가?**

💡 **쉽게 말하면**

- 같은 문제로 공부하고 시험 보면 성적이 **부풀려짐**.
- 그래서 처음 보는 문제(시험용)를 따로 떼어 둠.

**실행 흐름**

1. seed 정하기
2. 20%를 시험용으로 · 품종 비율 유지
3. 모양과 품종 비율 확인

**먼저 예측** · 150개의 20%를 떼면 학습 · 시험은 몇 개씩? — 답은 코드 실행 결과 아래에

In [4]:
import numpy as np                                    # 품종 개수 세기
from sklearn.model_selection import train_test_split  # 학습 · 시험 분할

# 왜 · 분할 seed를 한 곳에 두어 같은 환경에서 같은 분할을 다시 얻기 위해
RANDOM_STATE = 42                                     # 섞는 방식 고정 (3장)
X_train, X_test, y_train, y_test = train_test_split(  # 순서: 학습 X · 시험 X · 학습 y · 시험 y
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)  # 20% 시험 · 품종 비율 유지
print("Train:", X_train.shape, y_train.shape)         # 학습용
print("Test :", X_test.shape, y_test.shape)           # 시험용
print("시험 품종 개수:", np.bincount(y_test))         # stratify → 10 · 10 · 10

Train: (120, 4) (120,)
Test : (30, 4) (30,)
시험 품종 개수: [10 10 10]


**결과** · 예측 정답은 학습 120 · 시험 30

- `Train: (120, 4) (120,)` — 150 × 0.8 = 120 → 공부용
- `Test : (30, 4) (30,)` — 150 × 0.2 = 30 → 처음 보는 시험 문제
- `시험 품종 개수: [10 10 10]` — `stratify=y` → 시험에도 품종이 1 : 1 : 1

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>test_size=0.2</code></td><td>시험 비율 20%</td><td>데이터가 수백 개 이하일 때 흔한 값</td></tr>
<tr><td><code>stratify=y</code></td><td>정답 비율을 학습 · 시험에 똑같이</td><td>분류 문제 (기본 선택)</td></tr>
<tr><td><code>random_state=42</code></td><td>섞는 방식 고정</td><td>결과를 다시 만들어야 할 때 (항상)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 반환 순서는 `X_train, X_test, y_train, y_test` → 순서를 바꾸면 조용히 틀린 결과
> - 시험 30개면 1개 차이가 정확도 0.033 차이 → 작은 차이로 모델 우열을 정하지 않기

> ✅ **한 줄 정리** · `train_test_split`으로 처음 보는 시험용을 떼어 두고, 분류에서는 `stratify=y`로 품종 비율을 유지한다.

**딥러닝 코드에서는**

- `random_split(dataset, [n_train, n_val])` — 딥러닝도 학습 · 검증을 먼저 나눔
- `torch.Generator().manual_seed(42)`로 나누는 방식을 고정

---
## 2. 첫 모델과 모델 교체

> **이 장의 흐름**
>
> - DecisionTree로 fit → predict → 정확도 → 정확도를 손으로 다시 계산
> - 그다음 같은 틀에 LogisticRegression · KNN을 끼워 비교

<img src="images/pfm1_pattern.png" alt="모든 scikit-learn 모델은 같은 세 단계" width="763">

**파이프라인 위치** · ④ 모델 학습 → ⑥ 평가

### 2-1. `A` DecisionTree로 첫 모델 · `fit()` · `predict()` · `accuracy_score` <sub>원본 셀 12 · 13</sub>

> 🎯 **`fit` · `predict` · `accuracy_score`는 각각 무엇을 하는가?**

💡 **쉽게 말하면**

- `fit`은 **공부**, `predict`는 **시험 답안 쓰기**, `accuracy_score`는 **채점**.
- 공부는 학습용으로만, 시험은 시험용으로만 함.

**실행 흐름**

1. 모델 만들기
2. 학습용으로 fit
3. 시험용으로 predict
4. 정답과 비교해 정확도

**먼저 예측** · 시험 30문제 중 몇 개를 맞힐까? — 답은 코드 실행 결과 아래에

In [5]:
from sklearn.metrics import accuracy_score       # 정확도 = 맞힌 비율
from sklearn.tree import DecisionTreeClassifier  # 결정 트리 분류기

# 왜 · 만들기 → 학습 → 예측 → 채점의 기본 4단계를 한 모델로 익히기 위해
dt = DecisionTreeClassifier(random_state=18)  # 1. 모델 생성 (원본 seed 18)
dt.fit(X_train, y_train)                      # 2. 학습: 학습용으로만 패턴을 익힘
pred = dt.predict(X_test)                     # 3. 예측: 처음 보는 시험용
acc = accuracy_score(y_test, pred)            # 4. 평가: 정답과 비교
print(f"DecisionTree 정확도: {acc:.3f}")      # 원본과 같은 형식
print("예측 앞 10개:", pred[:10])             # 품종 번호로 답함
print("정답 앞 10개:", y_test[:10])           # 비교용

DecisionTree 정확도: 0.967


예측 앞 10개: [0 2 1 1 0 1 0 0 2 1]
정답 앞 10개: [0 2 1 1 0 1 0 0 2 1]


**결과** · 예측 정답은 29개 (정확도 0.967)

- `DecisionTree 정확도: 0.967` — 30문제 중 29개 정답 → 29 ÷ 30 = 0.967
- `예측 앞 10개: [0 2 1 1 0 1 0 0 2 1]` — 모델은 품종 번호로 답함
- `정답 앞 10개: [0 2 1 1 0 1 0 0 2 1]` — 앞 10개는 모두 맞음 → 틀린 1개는 뒤쪽

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>fit(X_train, y_train)</code></td><td>문제 + 정답으로 공부</td><td>학습용만 (시험용을 넣으면 커닝)</td></tr>
<tr><td><code>predict(X_test)</code></td><td>문제만 주고 답을 받음</td><td>시험 · 실제 서비스</td></tr>
<tr><td><code>accuracy_score(정답, 예측)</code></td><td>맞힌 비율</td><td>클래스가 고른 분류의 첫 지표</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `fit`에 `X_test`를 넣으면 시험 문제로 공부한 셈 → 점수가 부풀려짐
> - `accuracy_score(y_test, pred)` 순서: 정답이 먼저

> ✅ **한 줄 정리** · 모델은 학습용으로 `fit`, 시험용으로 `predict`, 정답과 비교해 `accuracy_score`로 평가한다.

**딥러닝 코드에서는**

- 학습 루프 `loss.backward()` · `optimizer.step()` = `fit`
- `model.eval()` 후 `model(x).argmax(1)` = `predict`
- `(pred == y).float().mean()` = 정확도

### 2-2. `A` 정확도를 손으로 계산 · `pred == y_test` · `.mean()` <sub>원본 셀 14</sub>

> 🎯 **정확도 0.967은 어떤 계산으로 나오고, 언제 착시가 생기는가?**

💡 **쉽게 말하면**

- 정확도 = **맞힌 개수 ÷ 전체 개수**.
- 한쪽 품종이 대부분이면 그 품종만 찍어도 점수가 높게 나옴.

**실행 흐름**

1. 예측 == 정답으로 맞음 표시
2. 개수 · 평균 계산
3. 틀린 자리 찾기
4. 다 같은 답으로 찍은 점수와 비교

**먼저 예측** · 모두 0번(setosa)으로 찍으면 정확도는? — 답은 코드 실행 결과 아래에

In [6]:
# 왜 · accuracy_score가 하는 계산을 직접 해 보고 결과가 같은지 확인하기 위해
correct = pred == y_test                              # 문제마다 맞음(True) / 틀림(False)
print("맞힌 개수:", correct.sum(), "/", len(y_test))  # True는 1로 셈
print("정확도 직접 계산:", round(correct.mean(), 3))  # 평균 = 맞힌 비율
wrong = np.where(~correct)[0]                         # 틀린 자리 번호
print("틀린 자리:", wrong, "정답", y_test[wrong], "→ 예측", pred[wrong])  # 무엇을 헷갈렸나


# 왜 · 아무것도 배우지 않은 기준선과 비교해 정확도의 의미를 확인하기 위해
all_zero = np.zeros_like(y_test)  # 모두 0번 품종으로 찍기
print("모두 0으로 찍은 정확도:", round(accuracy_score(y_test, all_zero), 3))  # 기준선

맞힌 개수: 29 / 30
정확도 직접 계산: 0.967
틀린 자리: [25] 정답 [1] → 예측 [2]
모두 0으로 찍은 정확도: 0.333


**결과** · 예측 정답은 0.333 — 품종이 1 : 1 : 1이라 찍기 점수는 낮음

- `맞힌 개수: 29 / 30` — 30문제 중 29개
- `정확도 직접 계산: 0.967` — `accuracy_score`와 같은 값 = 맞힌 개수 ÷ 전체
- `틀린 자리: [25] 정답 [1] → 예측 [2]` — versicolor(1)를 virginica(2)로 헷갈림 · 두 품종은 꽃잎 크기가 겹침
- `모두 0으로 찍은 정확도: 0.333` — 찍기 기준선 → 0.967은 기준선보다 훨씬 높음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>정확도</td><td>맞힌 개수 ÷ 전체</td><td>클래스가 고를 때 (Iris)</td></tr>
<tr><td>찍기 기준선</td><td>가장 많은 클래스로 다 찍은 점수</td><td>정확도가 진짜 좋은지 비교할 때</td></tr>
<tr><td>정밀도 · 재현율 (파일별 4)</td><td>클래스별로 어떤 실수를 했나</td><td>불균형 데이터 · 실수 비용이 다를 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 데이터가 불균형하면 정확도가 착시를 만듦 (예: 양성 1%면 전부 음성으로 찍어도 0.99)
> - 정확도 하나로 끝내지 말고 찍기 기준선부터 계산

> ✅ **한 줄 정리** · 정확도는 맞힌 개수 ÷ 전체이며, 찍기 기준선과 비교해야 좋은 점수인지 알 수 있다.

**딥러닝 코드에서는**

- `acc = (logits.argmax(1) == labels).float().mean()` — 같은 계산을 텐서로
- 불균형 데이터는 F1 · 재현율도 함께 기록

### 2-3. `A` 모델 바꿔 끼우기 · `LogisticRegression` · `KNeighborsClassifier` <sub>원본 셀 15 ~ 19</sub>

> 🎯 **모델을 바꿀 때 코드에서 무엇만 달라지고, 결과는 어떻게 비교하는가?**

💡 **쉽게 말하면**

- scikit-learn 모델은 **모두 같은 손잡이**(`fit` · `predict`)를 가짐.
- 그래서 모델 이름 한 줄만 바꾸면 됨.

**실행 흐름**

1. 모델 3개를 사전에 담기
2. 같은 분할로 fit → predict → 정확도
3. 표로 정렬

**먼저 예측** · 트리 · 로지스틱 · KNN 중 시험 정확도 1등은? — 답은 코드 실행 결과 아래에

In [7]:
from sklearn.linear_model import LogisticRegression  # 확률 기반 분류
from sklearn.neighbors import KNeighborsClassifier   # 가까운 이웃 k개의 다수결

# 왜 · 같은 분할 · 같은 지표로 세 모델을 공정하게 비교하기 위해
models = {"DecisionTree": DecisionTreeClassifier(random_state=42),  # seed 42 트리
          "LogisticRegression": LogisticRegression(max_iter=2000, random_state=42),  # 반복 넉넉히
          "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)}         # 이웃 5개
results = []                                                        # (모델, 정확도)
for name, model in models.items():                                  # 손잡이가 같아 같은 코드로 반복
    model.fit(X_train, y_train)                                     # 학습
    results.append((name, accuracy_score(y_test, model.predict(X_test))))  # 예측 · 채점
# 1등부터
table = pd.DataFrame(results, columns=["모델", "정확도"]).sort_values("정확도", ascending=False)
print(table.round(3).to_string(index=False))                        # 비교 표

                모델   정확도
         KNN (k=5) 1.000
LogisticRegression 0.967
      DecisionTree 0.933


**결과** · 예측 정답은 KNN (1.000)

- `KNN (k=5) 1.000` — 30문제 전부 정답
- `LogisticRegression 0.967` — 1개 틀림
- `DecisionTree 0.933` — 2개 틀림 · 2-1의 seed 18 트리(0.967)와 seed만 다름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>DecisionTree</td><td>질문(조건)을 이어 붙여 나눔</td><td>규칙을 눈으로 보고 싶을 때 (파일별 6)</td></tr>
<tr><td>LogisticRegression</td><td>특성의 가중합 → 확률</td><td>확률 · 기준선 모델이 필요할 때 (파일별 5)</td></tr>
<tr><td>KNN</td><td>가장 가까운 이웃 k개의 다수결</td><td>특성 단위가 같고 데이터가 작을 때 (보강 1)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 18은 KNN을 `dt`라는 이름에 담고 'DecisionTree 정확도'라고 출력함 → 이름표 오류 (KNN 결과임)
> - 시험 30개에서 1.000 vs 0.967은 1문제 차이 → 우열은 교차검증(파일별 2)으로

> ✅ **한 줄 정리** · scikit-learn 모델은 `fit` · `predict` 손잡이가 같아서, 같은 분할에서 이름만 바꿔 공정하게 비교한다.

**딥러닝 코드에서는**

- `model = resnet18()` → `model = efficientnet_b0()` — 학습 루프는 그대로 두고 모델만 교체
- 비교할 때는 같은 데이터 분할 · 같은 seed

---
## 3. random_state

> **이 장의 흐름**
>
> - seed를 바꾸면 시험 문제가 바뀌어 정확도가 흔들리는 것을 봄
> - 그다음 분할과 모델 seed를 모두 고정하면 몇 번을 실행해도 같은 결과

<img src="images/pfm1_seed.png" alt="seed가 바뀌면 시험 문제가 바뀐다" width="763">

**파이프라인 위치** · ② 분할 · ④ 모델 학습 — 재현성

### 3-1. `A` seed를 바꾸면 시험 문제가 바뀜 · `random_state` · `for seed in` <sub>원본 셀 20 · 21</sub>

> 🎯 **`random_state` 없이 나누면 왜 실행할 때마다 정확도가 달라지는가?**

💡 **쉽게 말하면**

- `random_state`는 **어떤 문제가 시험에 나올지 섞는 방식**.
- 안 넣으면 실행할 때마다 다르게 섞여 시험 문제가 바뀜.

**실행 흐름**

1. seed 0 ~ 4로 다섯 번 분할
2. 분할마다 같은 트리 학습
3. 정확도 비교

**먼저 예측** · seed만 바꿔도 정확도가 달라질까? — 답은 코드 실행 결과 아래에

In [8]:
# 원본 셀 21은 seed 없이 실행 → 매번 다름. 여기서는 seed 0 ~ 4로 그 차이를 다시 만들 수 있게 보임
# 왜 · seed 없이 실행하면 결과가 흔들리는 원본 상황을 같은 환경에서 다시 만들 수 있게 하기 위해
accs = []                                                        # 분할마다 정확도
for seed in range(5):                                            # 섞는 방식 5가지
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed)  # seed만 바꿈
    tree = DecisionTreeClassifier(random_state=0).fit(Xa, ya)    # 모델 seed는 고정
    accs.append(round(accuracy_score(yb, tree.predict(Xb)), 3))  # 시험 정확도
print("seed 0 ~ 4 정확도:", accs)                                # 분할만 바뀌어도 흔들림
print(f"가장 낮음 {min(accs)} · 가장 높음 {max(accs)}")          # 흔들리는 폭

seed 0 ~ 4 정확도: [1.0, 0.967, 0.933, 0.967, 0.967]
가장 낮음 0.933 · 가장 높음 1.0


**결과** · 예측 정답은 달라짐 — 시험 30문제가 바뀌기 때문

- `seed 0 ~ 4 정확도: [1.0, 0.967, 0.933, 0.967, 0.967]` — 같은 모델인데 분할만 바꿔도 0.933 ~ 1.0
- `가장 낮음 0.933 · 가장 높음 1.0` — 시험 30개에서 2문제 차이 = 0.067

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>random_state</code> 없음</td><td>실행할 때마다 다른 분할 · 다른 결과</td><td>쓰지 않기 (원본 셀 21이 보여 준 문제)</td></tr>
<tr><td><code>random_state=정수</code></td><td>같은 버전에서 항상 같은 분할</td><td><strong>기본 선택</strong></td></tr>
<tr><td>여러 seed 평균 (종합 실습)</td><td>분할 운을 평균으로 줄임</td><td>작은 데이터에서 모델을 비교할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 좋은 점수가 나오는 seed를 골라 보고하면 안 됨 (운을 고른 것)
> - seed는 실험 전에 정하고 끝까지 바꾸지 않음

> ✅ **한 줄 정리** · `random_state`가 없으면 실행마다 시험 문제가 바뀌어 정확도가 흔들리므로, 항상 정수로 고정한다.

**딥러닝 코드에서는**

- `torch.manual_seed(42)` · `np.random.seed(42)` · `random.seed(42)` — 노트북 첫 셀에서 고정
- DataLoader `shuffle=True`도 seed가 없으면 매번 순서가 다름

### 3-2. `B` 분할 seed와 모델 seed · `random_state=42` · `random_state=18` <sub>원본 셀 13 · 16 · 22</sub>

> 🎯 **분할의 seed와 모델의 seed는 각각 무엇을 고정하는가?**

💡 **쉽게 말하면**

- 분할 seed는 **시험 문제**를, 모델 seed는 **트리가 질문을 고르는 순서**를 고정.
- 둘 다 고정해야 완전히 같은 결과.

**실행 흐름**

1. 같은 분할에서 모델 seed만 바꾸기
2. seed를 고정하고 세 번 반복
3. 원본 셀 22 재현 (stratify 없음)

**먼저 예측** · seed 18 트리와 seed 42 트리의 시험 정확도는 같을까? — 답은 코드 실행 결과 아래에

In [9]:
# 왜 · 분할은 그대로 두고 모델 seed만 바꿨을 때의 영향을 따로 보기 위해
for s in [18, 42]:                                                    # 원본 셀 13(18) · 16(42)
    t = DecisionTreeClassifier(random_state=s).fit(X_train, y_train)  # 분할은 같고 모델 seed만 다름
    print(f"모델 seed {s}: {accuracy_score(y_test, t.predict(X_test)):.3f}")  # 2-1 · 2-3 값


# 왜 · seed를 모두 고정하면 같은 환경에서 결과가 똑같은지 세 번 실행해 확인하기 위해
runs = []                                                    # 같은 코드 세 번
for _ in range(3):                                           # 반복 실행
    # 원본 셀 22 (stratify 없음)
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42)
    t = DecisionTreeClassifier(random_state=18).fit(Xa, ya)  # 모델 seed도 고정
    runs.append(f"{accuracy_score(yb, t.predict(Xb)):.3f}")  # 문자열로 모음
print("고정한 채 세 번:", runs)                              # 몇 번을 실행해도 같음

모델 seed 18: 0.967
모델 seed 42: 0.933
고정한 채 세 번: ['1.000', '1.000', '1.000']


**결과** · 예측 정답은 다름 — 0.967 vs 0.933

- `모델 seed 18: 0.967` — 2-1의 첫 트리
- `모델 seed 42: 0.933` — 2-3의 트리 · 특성을 고르는 동점 처리가 seed에 따라 달라짐
- `고정한 채 세 번: ['1.000', '1.000', '1.000']` — 분할 · 모델 seed를 모두 고정하면 같은 환경에서 항상 같음 (원본 셀 22)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>분할 seed</td><td>어떤 샘플이 시험에 가나</td><td><code>train_test_split</code> · <code>KFold(shuffle=True)</code></td></tr>
<tr><td>모델 seed</td><td>모델 안의 무작위 선택</td><td>트리 · 랜덤 포레스트 · 신경망 초기값</td></tr>
<tr><td>둘 다 고정</td><td>완전히 같은 결과</td><td>보고서 · 과제 제출 (항상)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 22는 `stratify`가 없어 1-3 분할과 다름 → 정확도 1.000을 2-1의 0.967과 비교하지 않기
> - 비교는 같은 분할에서만

> ✅ **한 줄 정리** · 분할 seed는 시험 문제를, 모델 seed는 모델 안의 무작위 선택을 고정하므로 둘 다 정해야 결과가 같다.

**딥러닝 코드에서는**

- 가중치 초기값 · dropout · 데이터 순서가 모두 무작위 → seed 하나로 전부 고정
- GPU 연산은 seed를 고정해도 끝자리가 다를 수 있음

---
## 4. 종합 실습 · Iris 모델 3개를 seed 5개로 비교

> **이 장의 흐름**
>
> - 1 ~ 3장을 한 번에 사용
> - seed 5개 × 모델 3개를 같은 틀로 돌려 평균 · 흔들림을 비교

<img src="images/pfm1_practice.png" alt="seed 5개 평균 정확도와 흔들림" width="643">

**파이프라인 위치** · ① 데이터 준비 → ② 분할 → ④ 학습 → ⑥ 평가 → ⑦ 모델 비교

> 🎯 **한 번의 분할에서 KNN이 1등이었는데, seed를 바꿔도 1등이고 그 차이는 믿을 만한가?**

💡 **쉽게 말하면**

- 시험을 한 번만 보면 **운**이 섞임.
- 시험지를 5번 바꿔 평균을 내면 실력에 가까워짐.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>seed 5개 · 모델 3개 준비</td><td>1-3 · 2-3</td></tr>
<tr><td>2</td><td>seed마다 stratify 분할</td><td>1-3 · 3-1</td></tr>
<tr><td>3</td><td>모델마다 fit → predict → 정확도</td><td>2-1 · 2-2</td></tr>
<tr><td>4</td><td>평균 · 표준편차 표</td><td>pandas</td></tr>
<tr><td>5</td><td>seed 42 한 번 결과와 비교</td><td>2-3</td></tr>
</tbody></table>

**먼저 예측** · seed 5개 평균에서도 KNN이 1등일까? 2등과의 차이는 흔들림(표준편차)보다 클까? — 답은 코드 실행 결과 아래에

In [10]:
# 1단계: seed 5개 · 모델 3개 (모델은 함수로 만들어 seed마다 새로 생성)
# 왜 · 한 번의 분할 운에 기대지 않고 여러 seed로 모델을 비교하기 위해
SEEDS = [0, 1, 2, 3, 42]                                                    # 시험지 5종류
MAKERS = {"DecisionTree": lambda: DecisionTreeClassifier(random_state=42),  # 매번 새 모델
          # 같은 설정
          "LogisticRegression": lambda: LogisticRegression(max_iter=2000, random_state=42),
          "KNN (k=5)": lambda: KNeighborsClassifier(n_neighbors=5)}         # 같은 설정


# 2 · 3단계: seed마다 분할 → 모델마다 학습 · 채점
# 왜 · 모든 모델이 seed마다 같은 분할을 쓰게 해 공정하게 비교하기 위해
rows = []                                                             # (seed, 모델, 정확도)
for seed in SEEDS:                                                    # 시험지마다
    # 품종 비율 유지
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    for name, make in MAKERS.items():                                 # 모델마다
        m = make().fit(Xa, ya)                                        # 새 모델로 학습
        rows.append((seed, name, accuracy_score(yb, m.predict(Xb))))  # 시험 정확도
scores = pd.DataFrame(rows, columns=["seed", "모델", "정확도"])       # 15행 표
# seed × 모델
print(scores.pivot(index="seed", columns="모델", values="정확도").round(3).to_string())

모델    DecisionTree  KNN (k=5)  LogisticRegression
seed                                             
0            0.967      1.000               1.000
1            0.967      0.967               0.967
2            0.967      0.967               1.000
3            0.900      0.933               0.900
42           0.933      1.000               0.967


**실행 흐름**

1. seed별 결과를 모델마다 평균 · 표준편차로 요약
2. seed 42 한 번의 결과와 나란히 비교

In [11]:
# 4단계: 모델별 평균 · 표준편차
# 왜 · 평균과 흔들림을 함께 봐야 우연이 아닌 차이인지 판단할 수 있기 때문에
# 평균 1등부터
summary = scores.groupby("모델")["정확도"].agg(["mean", "std"]).sort_values("mean", ascending=False)
print(summary.round(3).to_string())  # 평균 · 흔들림


# 5단계: seed 42 한 번의 결과와 비교
# 왜 · 한 번의 결과만 보면 결론이 달라질 수 있음을 평균과 비교하기 위해
one = scores[scores["seed"] == 42].set_index("모델")["정확도"]              # 2-3과 같은 분할
print("seed 42 한 번 1등:", one.idxmax(), "· 평균 1등:", summary.index[0])  # 같은가?

                     mean    std
모델                              
KNN (k=5)           0.973  0.028
LogisticRegression  0.967  0.041
DecisionTree        0.947  0.030
seed 42 한 번 1등: KNN (k=5) · 평균 1등: KNN (k=5)


**결과** · 예측 정답은 1등은 KNN 그대로 · 하지만 2등과 차이 0.006은 표준편차 0.028보다 훨씬 작음

- `42           0.933      1.000               0.967` — seed 42 줄은 2-3과 같은 값
- `3            0.900      0.933               0.900` — seed 3 시험지는 어려워 세 모델 모두 떨어짐 = 분할 운
- `0            0.967      1.000               1.000` — seed 0에서는 로지스틱도 1.000 → 순위가 seed마다 바뀜
- `KNN (k=5)           0.973  0.028` — 평균 1등 · 흔들림(표준편차) 0.028
- `LogisticRegression  0.967  0.041` — 평균 2등 · 1등과 차이 0.006 < 표준편차 → 사실상 같음
- `DecisionTree        0.947  0.030` — 평균 3등
- `seed 42 한 번 1등: KNN (k=5) · 평균 1등: KNN (k=5)` — 이번엔 같지만, 차이가 작아 데이터가 바뀌면 뒤집힐 수 있음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 번 분할 (2-3)</td><td>빠르지만 분할 운이 섞임</td><td>코드가 도는지 확인할 때</td></tr>
<tr><td>seed 여러 개 평균 (이 장)</td><td>운을 줄인 비교 · 표준편차로 흔들림까지</td><td>작은 데이터에서 모델 비교</td></tr>
<tr><td>교차검증 (파일별 2)</td><td>모든 샘플이 한 번씩 시험에 나옴</td><td><strong>모델 선택의 기본</strong></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - seed 여러 개로 나온 평균에서 1등을 고른 뒤, 그 점수를 최종 성적으로 보고하지 않기
> - 최종 성적은 따로 떼어 둔 test로 (파일별 2 · 5)

> ✅ **한 줄 정리** · 한 번의 분할 결과는 운이 섞이므로 seed 여러 개의 평균과 흔들림으로 비교하고, 흔들림보다 작은 차이는 같다고 본다.

**딥러닝 코드에서는**

- seed 3개로 학습해 평균 ± 표준편차를 보고 — 논문 표의 `0.912 ± 0.004`
- 한 번의 학습 곡선으로 모델 우열을 정하지 않기

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. X · y</td><td>X(150, 4) = 측정값 4개, y(150,) = 품종 번호, 품종마다 50송이</td><td>1-1 · 1-2</td></tr>
<tr><td>2. 분할 이유</td><td>학습 120 · 시험 30 (stratify로 10 · 10 · 10)</td><td>1-3</td></tr>
<tr><td>3. fit → predict → score</td><td>트리 0.967 = 29 ÷ 30, 찍기 기준선 0.333</td><td>2-1 · 2-2</td></tr>
<tr><td>4. 모델 교체</td><td>KNN 1.000 · 로지스틱 0.967 · 트리 0.933 (같은 분할)</td><td>2-3</td></tr>
<tr><td>5. random_state</td><td>seed만 바꿔도 0.933 ~ 1.0 · 고정하면 세 번 모두 1.000 · 평균 차이 0.006 &lt; 표준편차</td><td>3-1 · 3-2 · 4</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>X · y</td><td>문제지(150, 4)와 정답지(150,)</td><td><code>images, labels</code></td></tr>
<tr><td><code>A</code></td><td>분할</td><td>시험용은 공부에 쓰지 않음</td><td><code>random_split</code> · val loader</td></tr>
<tr><td><code>A</code></td><td>fit · predict</td><td>공부 · 답안 쓰기 · 채점</td><td>학습 루프 · <code>model.eval()</code></td></tr>
<tr><td><code>A</code></td><td>정확도</td><td>맞힌 개수 ÷ 전체 · 찍기 기준선과 비교</td><td><code>(pred == y).float().mean()</code></td></tr>
<tr><td><code>A</code></td><td>random_state</td><td>분할 · 모델 seed 모두 고정</td><td><code>torch.manual_seed(42)</code></td></tr>
<tr><td><code>B</code></td><td>stratify</td><td>정답 비율 유지</td><td>클래스별 샘플링</td></tr>
<tr><td><code>B</code></td><td>모델 교체</td><td>손잡이가 같아 이름만 바꿈</td><td>백본 교체</td></tr>
<tr><td><code>B</code></td><td>seed 평균</td><td>운을 줄인 비교</td><td><code>평균 ± 표준편차</code> 보고</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · `load_iris(return_X_y=True)`의 X · y shape는?
- **1-2** · Iris 품종별 샘플 수는?
- **1-3** · `stratify=y`는 무엇을 하는가?
- **2-1** · `fit` · `predict` · `accuracy_score`의 역할은?
- **2-2** · 정확도의 계산식과 착시가 생기는 경우는?
- **2-3** · 모델을 바꿀 때 코드에서 달라지는 것은?
- **3-1** · `random_state`가 없으면?
- **3-2** · 분할 seed와 모델 seed의 차이는?
- **4** · 한 번의 분할 1등을 믿으면 안 되는 이유는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `test_size=0.3`, `stratify=y`, `random_state=7`로 나눈 뒤 학습 · 시험 개수 출력
2. 그 분할에서 `KNeighborsClassifier(n_neighbors=3)`의 시험 정확도를 소수 셋째 자리로 출력

In [12]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-3에서 KNN의 `n_neighbors`를 1 · 15로 바꾸면 정확도는? (원본 목표의 '파라미터 변경')
2. 1-3에서 `stratify=y`를 빼면 시험 품종 개수는 여전히 10 · 10 · 10일까?
3. 4장 종합 실습에 `DecisionTreeClassifier(max_depth=2)`를 넣으면 평균 순위는?

In [13]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · X `(150, 4)` · y `(150,)`
- **1-2** · 50 · 50 · 50 (균형)
- **1-3** · 학습 · 시험의 정답 비율을 전체와 같게 유지
- **2-1** · 학습용으로 공부 · 시험용 답안 · 정답과 비교해 맞힌 비율
- **2-2** · 맞힌 개수 ÷ 전체 · 클래스가 불균형하면 찍기만 해도 높음
- **2-3** · 모델을 만드는 한 줄만 (fit · predict는 같음)
- **3-1** · 실행마다 분할이 달라져 정확도가 흔들림
- **3-2** · 시험 문제 고정 vs 모델 안 무작위 선택 고정
- **4** · 분할 운이 섞임 → seed 평균 · 교차검증으로 비교, 표준편차보다 작은 차이는 같다고 봄

#### 코드 문제 답

```python
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, stratify=y, random_state=7)
print(len(ya), len(yb))  # 105 45
knn3 = KNeighborsClassifier(n_neighbors=3).fit(Xa, ya)
print(round(accuracy_score(yb, knn3.predict(Xb)), 3))  # 0.956
```

#### 변형 문제 답

*문제 → 답*

- **1** · k=1 · k=15 모두 0.967 (seed 42 분할) — k=5의 1.000보다 1문제 낮음 · 최적의 k는 보강 1에서 교차검증으로
- **2** · 아님 — `[10 9 11]`처럼 조금 달라짐 (seed 42 기준)
- **3** · seed별 0.933 · 0.933 · 0.933 · 0.833 · 0.933 → 평균 0.913으로 가장 낮음 — 질문 2번뿐이라 versicolor · virginica를 덜 가름

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) <code>fit(X_test, y_test)</code>로 학습함</td><td>학습 · 시험 역할을 헷갈림</td><td>공부는 학습용, 시험은 시험용</td><td>fit 괄호 안에 test가 있나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 학습 데이터와 테스트 데이터를 왜 나누나요?**
> - 같은 데이터로 학습하고 평가하면 모델이 외운 문제로 시험을 보는 셈이라 성능이 부풀려집니다.
> - 처음 보는 테스트 데이터로 평가해야 새 데이터에서의 성능을 추정할 수 있습니다.
> - 분류에서는 stratify로 정답 비율을 유지합니다.

**Q2. scikit-learn에서 모델을 바꾸기 쉬운 이유는 무엇인가요?**
> - 모든 모델이 fit, predict 같은 같은 메서드를 가지고 있어서 모델을 만드는 한 줄만 바꾸면 됩니다.
> - 그래서 같은 분할에서 여러 모델을 공정하게 비교할 수 있습니다.

**Q3. random_state는 왜 고정하나요?**
> - 데이터를 섞는 방식과 모델 안의 무작위 선택이 실행마다 달라지면 결과를 다시 만들 수 없습니다.
> - Iris에서도 seed만 바꿔 정확도가 0.933에서 1.0까지 흔들렸습니다.
> - 그래서 분할과 모델 seed를 모두 고정하고, 비교는 여러 seed 평균이나 교차검증으로 합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>load_iris(return_X_y=True)</code></td><td>sklearn.datasets</td><td>Iris X · y 받기</td><td>1-1 <code>X, y = load_iris(return_X_y=True)</code></td></tr>
<tr><td><code>.shape</code> · <code>np.bincount()</code></td><td>numpy</td><td>모양 · 클래스 개수</td><td>1-1 <code>X.shape</code> · 1-3 <code>np.bincount(y_test)</code></td></tr>
<tr><td><code>pd.DataFrame</code> · <code>value_counts()</code></td><td>pandas</td><td>표로 보기 · 개수 세기</td><td>1-2 <code>iris_df["label"].value_counts()</code></td></tr>
<tr><td><code>train_test_split</code></td><td>sklearn.model_selection</td><td>학습 · 시험 분할</td><td>1-3 <code>train_test_split(X, y, test_size=0.2, …)</code></td></tr>
<tr><td><code>DecisionTreeClassifier</code></td><td>sklearn.tree</td><td>결정 트리</td><td>2-1 <code>DecisionTreeClassifier(random_state=18)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>fit()</code> · <code>predict()</code></td><td>모든 sklearn 모델</td><td>학습 · 예측</td><td>2-1 <code>dt.fit(X_train, y_train)</code></td></tr>
<tr><td><code>accuracy_score</code></td><td>sklearn.metrics</td><td>정확도</td><td>2-1 <code>accuracy_score(y_test, pred)</code></td></tr>
<tr><td><code>LogisticRegression</code></td><td>sklearn.linear_model</td><td>로지스틱 회귀</td><td>2-3 <code>LogisticRegression(max_iter=2000, …)</code></td></tr>
<tr><td><code>KNeighborsClassifier</code></td><td>sklearn.neighbors</td><td>KNN</td><td>2-3 <code>KNeighborsClassifier(n_neighbors=5)</code></td></tr>
<tr><td><code>groupby().agg()</code></td><td>pandas</td><td>모델별 평균 · 표준편차</td><td>4 <code>scores.groupby("모델")["정확도"].agg(…)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>시험용은 마지막 채점에만</td><td><code>fit</code>에 <code>X_test</code> 넣기</td></tr>
<tr><td><code>stratify=y</code>로 비율 유지</td><td>작은 시험에서 0.033 차이로 우열 정하기</td></tr>
<tr><td>seed를 처음에 정하고 고정</td><td>좋은 점수가 나온 seed 골라 보고</td></tr>
<tr><td>찍기 기준선과 비교</td><td>정확도 하나로 결론</td></tr>
<tr><td>모델 이름표와 결과 맞추기</td><td>KNN을 <code>dt</code>에 담고 트리라고 출력</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>X</code> vs <code>y</code></td><td>문제(측정값) vs 정답(품종)</td><td>fit에는 둘 다, predict에는 X만</td></tr>
<tr><td><code>train</code> vs <code>test</code></td><td>공부용 vs 처음 보는 시험</td><td>test는 마지막에 한 번</td></tr>
<tr><td>분할 seed vs 모델 seed</td><td>시험 문제 vs 모델 안 무작위</td><td>둘 다 고정</td></tr>
<tr><td>한 번 분할 vs seed 평균</td><td>학습 1번 · 운 섞임 vs 여러 번 학습 · 운 줄임</td><td>비교는 평균 · 교차검증</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · 2-1 세 단계를 빈 셀에 직접</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>4장 종합 실습을 빈 셀에 처음부터 다시 짜기</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 파일별 2(펭귄 교차검증) 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 `02_iris_quickstart.ipynb`의 셀 23개를 순서대로 1 ~ 3장 · 8개 섹션으로 나눔
- 원본 셀 6 · 7(같은 코드 두 번) · 17(빈 셀)은 1-1에 합침
- 원본 셀 21(seed 없음)은 두 번 실행해도 같도록 seed 0 ~ 4 비교로 바꿈 (보여 주려는 내용은 같음)
- 원본 산점도(셀 9)는 1장 그림으로 · 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>셀 0</td><td>목표 4개</td><td>0. 원본 목표</td></tr>
<tr><td>셀 1 · 2</td><td>버전 확인</td><td>0. 준비 코드</td></tr>
<tr><td>셀 3 · 19</td><td>fit → predict → score 3줄 · 성능 올리기는 어려움</td><td>0. 원본 힌트 · 2장 그림</td></tr>
<tr><td>셀 4 ~ 7</td><td>왜 Iris · X · y · 묶음</td><td>1-1</td></tr>
<tr><td>셀 8 · 9</td><td>DataFrame · 산점도</td><td>1-2 · 1장 그림</td></tr>
<tr><td>셀 10 · 11</td><td>Train / Test 분리</td><td>1-3</td></tr>
<tr><td>셀 12 · 13</td><td>첫 모델 DecisionTree</td><td>2-1</td></tr>
<tr><td>셀 14</td><td>정확도 해석</td><td>2-2</td></tr>
<tr><td>셀 15 ~ 18</td><td>모델 바꿔 보기</td><td>2-3</td></tr>
<tr><td>셀 20 · 21</td><td>random_state란 · seed 없이</td><td>3-1</td></tr>
<tr><td>셀 22</td><td>seed 고정</td><td>3-2</td></tr>
<tr><td>(새로 추가)</td><td>seed 5개 평균 비교</td><td>4. 종합 실습</td></tr>
</tbody></table>

### 전체 코드

1 ~ 4장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 iris 불러오기 · 모양 확인
2. 1-2 표로 보기
3. 1-3 학습 · 평가 분할

In [14]:
from sklearn.datasets import load_iris                # 내장 Iris 데이터
import pandas as pd                                   # 표 만들기
import numpy as np                                    # 품종 개수 세기
from sklearn.model_selection import train_test_split  # 학습 · 시험 분할


# ---------- 1-1 ----------
# 왜 · 데이터 불러오기 · 확인을 본문 순서대로 다시 실행하기 위해

# 왜 · 모델에 바로 넣을 특성 X와 정답 y를 한 번에 받기 위해
X, y = load_iris(return_X_y=True)  # 문제(X)와 정답(y)만 바로 받기
print("X shape:", X.shape)         # (샘플 수, 특성 수)
print("y shape:", y.shape)         # 샘플마다 정답 1개

# 왜 · 숫자뿐 아니라 열 이름 · 클래스 이름 같은 설명 정보도 함께 보기 위해
iris = load_iris()                                       # 전체 묶음(Bunch) · 원본 셀 6 · 7
print("열쇠:", list(iris.keys())[:6])                    # 원본은 묶음 전체를 출력 → 열쇠만
print("첫 꽃:", iris.data[0], "→ 품종", iris.target[0])  # 한 샘플 = 측정값 4개 + 정답


# ---------- 1-2 ----------

# 왜 · 숫자 라벨이 어떤 품종인지 표로 확인해 결과를 사람이 읽을 수 있게 하기 위해
print("iris target명:", iris.target_names)                          # 0 · 1 · 2의 이름
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)  # 측정값 4열 표
iris_df["label"] = iris.target                                      # 정답 열 추가
print(iris_df.head(3).to_string())                                  # 앞 3행
print(iris_df["label"].value_counts().sort_index().to_dict())       # 품종별 개수


# ---------- 1-3 ----------

# 왜 · 분할 seed를 한 곳에 두어 같은 환경에서 같은 분할을 다시 얻기 위해
RANDOM_STATE = 42                                     # 섞는 방식 고정 (3장)
X_train, X_test, y_train, y_test = train_test_split(  # 순서: 학습 X · 시험 X · 학습 y · 시험 y
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)  # 20% 시험 · 품종 비율 유지
print("Train:", X_train.shape, y_train.shape)         # 학습용
print("Test :", X_test.shape, y_test.shape)           # 시험용
print("시험 품종 개수:", np.bincount(y_test))         # stratify → 10 · 10 · 10

X shape: (150, 4)
y shape: (150,)
열쇠: ['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names']
첫 꽃: [5.1 3.5 1.4 0.2] → 품종 0
iris target명: ['setosa' 'versicolor' 'virginica']
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  label
0                5.1               3.5                1.4               0.2      0
1                4.9               3.0                1.4               0.2      0
2                4.7               3.2                1.3               0.2      0
{0: 50, 1: 50, 2: 50}
Train: (120, 4) (120,)
Test : (30, 4) (30,)
시험 품종 개수: [10 10 10]


**실행 흐름**

1. 2-1 결정 트리 학습 · 예측
2. 2-2 정확도 직접 계산 · 기준선
3. 2-3 세 모델 비교

In [15]:
from sklearn.metrics import accuracy_score           # 정확도 = 맞힌 비율
from sklearn.tree import DecisionTreeClassifier      # 결정 트리 분류기
from sklearn.linear_model import LogisticRegression  # 확률 기반 분류
from sklearn.neighbors import KNeighborsClassifier   # 가까운 이웃 k개의 다수결


# ---------- 2-1 ----------
# 왜 · 분할 · 학습 · 채점 · 비교를 본문 순서대로 다시 실행하기 위해

# 왜 · 만들기 → 학습 → 예측 → 채점의 기본 4단계를 한 모델로 익히기 위해
dt = DecisionTreeClassifier(random_state=18)  # 1. 모델 생성 (원본 seed 18)
dt.fit(X_train, y_train)                      # 2. 학습: 학습용으로만 패턴을 익힘
pred = dt.predict(X_test)                     # 3. 예측: 처음 보는 시험용
acc = accuracy_score(y_test, pred)            # 4. 평가: 정답과 비교
print(f"DecisionTree 정확도: {acc:.3f}")      # 원본과 같은 형식
print("예측 앞 10개:", pred[:10])             # 품종 번호로 답함
print("정답 앞 10개:", y_test[:10])           # 비교용


# ---------- 2-2 ----------
# 왜 · accuracy_score가 하는 계산을 직접 해 보고 결과가 같은지 확인하기 위해
correct = pred == y_test                              # 문제마다 맞음(True) / 틀림(False)
print("맞힌 개수:", correct.sum(), "/", len(y_test))  # True는 1로 셈
print("정확도 직접 계산:", round(correct.mean(), 3))  # 평균 = 맞힌 비율
wrong = np.where(~correct)[0]                         # 틀린 자리 번호
print("틀린 자리:", wrong, "정답", y_test[wrong], "→ 예측", pred[wrong])  # 무엇을 헷갈렸나


# 왜 · 아무것도 배우지 않은 기준선과 비교해 정확도의 의미를 확인하기 위해
all_zero = np.zeros_like(y_test)  # 모두 0번 품종으로 찍기
print("모두 0으로 찍은 정확도:", round(accuracy_score(y_test, all_zero), 3))  # 기준선


# ---------- 2-3 ----------

# 왜 · 같은 분할 · 같은 지표로 세 모델을 공정하게 비교하기 위해
models = {"DecisionTree": DecisionTreeClassifier(random_state=42),  # seed 42 트리
          "LogisticRegression": LogisticRegression(max_iter=2000, random_state=42),  # 반복 넉넉히
          "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)}         # 이웃 5개
results = []                                                        # (모델, 정확도)
for name, model in models.items():                                  # 손잡이가 같아 같은 코드로 반복
    model.fit(X_train, y_train)                                     # 학습
    results.append((name, accuracy_score(y_test, model.predict(X_test))))  # 예측 · 채점
# 1등부터
table = pd.DataFrame(results, columns=["모델", "정확도"]).sort_values("정확도", ascending=False)
print(table.round(3).to_string(index=False))                        # 비교 표

DecisionTree 정확도: 0.967
예측 앞 10개: [0 2 1 1 0 1 0 0 2 1]
정답 앞 10개: [0 2 1 1 0 1 0 0 2 1]
맞힌 개수: 29 / 30
정확도 직접 계산: 0.967
틀린 자리: [25] 정답 [1] → 예측 [2]
모두 0으로 찍은 정확도: 0.333
                모델   정확도
         KNN (k=5) 1.000
LogisticRegression 0.967
      DecisionTree 0.933


**실행 흐름**

1. 3-1 seed에 따른 분할 차이
2. 3-2 모델 seed · 모두 고정한 결과

In [16]:
# ---------- 3-1 ----------
# 원본 셀 21은 seed 없이 실행 → 매번 다름. 여기서는 seed 0 ~ 4로 그 차이를 다시 만들 수 있게 보임
# 왜 · seed 없이 실행하면 결과가 흔들리는 원본 상황을 같은 환경에서 다시 만들 수 있게 하기 위해
accs = []                                                        # 분할마다 정확도
for seed in range(5):                                            # 섞는 방식 5가지
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed)  # seed만 바꿈
    tree = DecisionTreeClassifier(random_state=0).fit(Xa, ya)    # 모델 seed는 고정
    accs.append(round(accuracy_score(yb, tree.predict(Xb)), 3))  # 시험 정확도
print("seed 0 ~ 4 정확도:", accs)                                # 분할만 바뀌어도 흔들림
print(f"가장 낮음 {min(accs)} · 가장 높음 {max(accs)}")          # 흔들리는 폭


# ---------- 3-2 ----------
# 왜 · 분할은 그대로 두고 모델 seed만 바꿨을 때의 영향을 따로 보기 위해
for s in [18, 42]:                                                    # 원본 셀 13(18) · 16(42)
    t = DecisionTreeClassifier(random_state=s).fit(X_train, y_train)  # 분할은 같고 모델 seed만 다름
    print(f"모델 seed {s}: {accuracy_score(y_test, t.predict(X_test)):.3f}")  # 2-1 · 2-3 값


# 왜 · seed를 모두 고정하면 같은 환경에서 결과가 똑같은지 세 번 실행해 확인하기 위해
runs = []                                                    # 같은 코드 세 번
for _ in range(3):                                           # 반복 실행
    # 원본 셀 22 (stratify 없음)
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42)
    t = DecisionTreeClassifier(random_state=18).fit(Xa, ya)  # 모델 seed도 고정
    runs.append(f"{accuracy_score(yb, t.predict(Xb)):.3f}")  # 문자열로 모음
print("고정한 채 세 번:", runs)                              # 몇 번을 실행해도 같음

seed 0 ~ 4 정확도: [1.0, 0.967, 0.933, 0.967, 0.967]
가장 낮음 0.933 · 가장 높음 1.0
모델 seed 18: 0.967
모델 seed 42: 0.933
고정한 채 세 번: ['1.000', '1.000', '1.000']


**실행 흐름**

1. seed 5개 × 모델 3개로 학습 · 채점
2. 모델별 평균 · 표준편차
3. seed 42 한 번과 비교

In [17]:
# ---------- 4. 종합 실습 ----------
# 1단계: seed 5개 · 모델 3개 (모델은 함수로 만들어 seed마다 새로 생성)
# 왜 · 한 번의 분할 운에 기대지 않고 여러 seed로 모델을 비교하기 위해
SEEDS = [0, 1, 2, 3, 42]                                                    # 시험지 5종류
MAKERS = {"DecisionTree": lambda: DecisionTreeClassifier(random_state=42),  # 매번 새 모델
          # 같은 설정
          "LogisticRegression": lambda: LogisticRegression(max_iter=2000, random_state=42),
          "KNN (k=5)": lambda: KNeighborsClassifier(n_neighbors=5)}         # 같은 설정


# 2 · 3단계: seed마다 분할 → 모델마다 학습 · 채점
# 왜 · 모든 모델이 seed마다 같은 분할을 쓰게 해 공정하게 비교하기 위해
rows = []                                                             # (seed, 모델, 정확도)
for seed in SEEDS:                                                    # 시험지마다
    # 품종 비율 유지
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    for name, make in MAKERS.items():                                 # 모델마다
        m = make().fit(Xa, ya)                                        # 새 모델로 학습
        rows.append((seed, name, accuracy_score(yb, m.predict(Xb))))  # 시험 정확도
scores = pd.DataFrame(rows, columns=["seed", "모델", "정확도"])       # 15행 표
# seed × 모델
print(scores.pivot(index="seed", columns="모델", values="정확도").round(3).to_string())


# 4단계: 모델별 평균 · 표준편차
# 왜 · 평균과 흔들림을 함께 봐야 우연이 아닌 차이인지 판단할 수 있기 때문에
# 평균 1등부터
summary = scores.groupby("모델")["정확도"].agg(["mean", "std"]).sort_values("mean", ascending=False)
print(summary.round(3).to_string())  # 평균 · 흔들림


# 5단계: seed 42 한 번의 결과와 비교
# 왜 · 한 번의 결과만 보면 결론이 달라질 수 있음을 평균과 비교하기 위해
one = scores[scores["seed"] == 42].set_index("모델")["정확도"]              # 2-3과 같은 분할
print("seed 42 한 번 1등:", one.idxmax(), "· 평균 1등:", summary.index[0])  # 같은가?

모델    DecisionTree  KNN (k=5)  LogisticRegression
seed                                             
0            0.967      1.000               1.000
1            0.967      0.967               0.967
2            0.967      0.967               1.000
3            0.900      0.933               0.900
42           0.933      1.000               0.967
                     mean    std
모델                              
KNN (k=5)           0.973  0.028
LogisticRegression  0.967  0.041
DecisionTree        0.947  0.030
seed 42 한 번 1등: KNN (k=5) · 평균 1등: KNN (k=5)
